# Prithvi Mac End-to-End Smoke Test

This notebook tests the real Mac-local foundation-model path:

- load real FTW Africa samples,
- visualize image and mask tensors,
- adapt FTW PlanetScope windows into Prithvi's expected tensor shape,
- load the real `Prithvi-EO-2.0 300M` checkpoint,
- run a forward pass through the frozen backbone,
- optionally run a one-batch segmentation-head training pass.

This is a pipeline validation notebook. The FTW-to-Prithvi adapter is pragmatic because FTW uses PlanetScope bands while Prithvi was pretrained on HLS bands. Use this to validate engineering before the GCP experiment phase.

## Environment

In [ ]:
import os

os.environ.setdefault('PYTORCH_ENABLE_MPS_FALLBACK', '1')

## Imports

In [ ]:
from pathlib import Path
import json
import subprocess
import sys
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from torch.nn import functional as F
from torch.utils.data import DataLoader

from eval_harness.config import dataset_config_path, load_yaml
from eval_harness.datasets import build_dataset
from eval_harness.models.prithvi import (
    adapt_ftw_batch_to_prithvi,
    load_prithvi_eo_v2_300m,
    prithvi_feature_map,
)
from eval_harness.training.local_pilot import collate_segmentation

## Device Check

In [ ]:
device = 'mps' if torch.backends.mps.is_available() else 'cpu'

device_info = {
    'torch_version': torch.__version__,
    'mps_built': torch.backends.mps.is_built(),
    'mps_available': torch.backends.mps.is_available(),
    'selected_device': device,
}

device_info

If `mps_available` is `False`, the notebook will still run on CPU for small smoke tests, but your Mac GPU is not active in this kernel. In that case, restart VS Code/Jupyter with the `climate-foundation-models` kernel and re-run this cell before doing full `224x224` forward tests.

## Dataset Config

In [ ]:
dataset_name = 'ftw_africa'
dataset_cfg = load_yaml(dataset_config_path(dataset_name))

display(pd.DataFrame([{
    'dataset': dataset_cfg['name'],
    'display_name': dataset_cfg['display_name'],
    'region': dataset_cfg['region'],
    'task_type': dataset_cfg['task_type'],
    'num_classes': dataset_cfg['num_classes'],
    'input_shape': tuple(dataset_cfg['input']['shape']),
    'loader': dataset_cfg['data']['loader'],
}]))

## Load Real FTW Data

In [ ]:
train_ds = build_dataset(dataset_cfg, split='train')
val_ds = build_dataset(dataset_cfg, split='val')

split_rows = []
for split_name, ds in [('train', train_ds), ('val', val_ds)]:
    countries = [item.get('country') for item in getattr(ds, 'items', [])]
    split_rows.append({
        'split': split_name,
        'n': len(ds),
        'countries': dict(pd.Series(countries).value_counts().sort_index()) if countries else None,
    })

display(pd.DataFrame(split_rows))

## Inspect One Batch

In [ ]:
loader = DataLoader(
    train_ds,
    batch_size=1,
    shuffle=False,
    num_workers=0,
    collate_fn=lambda batch: collate_segmentation(batch, image_size=256),
)

batch = next(iter(loader))

{
    'id': batch['id'][0],
    'country': batch['country'][0],
    'x_shape': tuple(batch['x'].shape),
    'x_min': float(batch['x'].min()),
    'x_mean': float(batch['x'].mean()),
    'x_max': float(batch['x'].max()),
    'y_shape': tuple(batch['y'].shape),
    'mask_classes': sorted(torch.unique(batch['y']).tolist()),
}

## Visualize FTW Sample

In [ ]:
x = batch['x'][0].numpy()
y = batch['y'][0].numpy()

rgb_a = np.moveaxis(np.clip(x[[2, 1, 0]] * 3.0, 0, 1), 0, -1)
rgb_b = np.moveaxis(np.clip(x[[6, 5, 4]] * 3.0, 0, 1), 0, -1)

fig, axes = plt.subplots(1, 3, figsize=(12, 4))
axes[0].imshow(rgb_a)
axes[0].set_title('FTW window A RGB')
axes[1].imshow(rgb_b)
axes[1].set_title('FTW window B RGB')
axes[2].imshow(y, vmin=0, vmax=2, cmap='viridis')
axes[2].set_title('Mask: background/interior/boundary')
for ax in axes:
    ax.axis('off')
plt.tight_layout()

## Adapt FTW Batch To Prithvi Input

In [ ]:
PRITHVI_IMAGE_SIZE = 224

x_prithvi = adapt_ftw_batch_to_prithvi(batch['x'].to(device), image_size=PRITHVI_IMAGE_SIZE)

{
    'prithvi_input_shape': tuple(x_prithvi.shape),
    'expected': '(batch, 6 HLS-like bands, 4 frames, 224, 224)',
    'min': float(x_prithvi.min().cpu()),
    'mean': float(x_prithvi.mean().cpu()),
    'max': float(x_prithvi.max().cpu()),
}

## Load Real Prithvi Checkpoint

In [ ]:
started = time.perf_counter()
prithvi = load_prithvi_eo_v2_300m(load_weights=True, device=device, freeze=True)
load_seconds = time.perf_counter() - started

{
    'model': 'prithvi_eo_v2_300m',
    'device': device,
    'load_seconds': load_seconds,
    'checkpoint_path': str(prithvi.checkpoint_path),
    'missing_keys': prithvi.missing_keys[:10],
    'unexpected_keys': prithvi.unexpected_keys[:10],
}

## Forward Pass

In [ ]:
started = time.perf_counter()
with torch.no_grad():
    feature_map = prithvi_feature_map(prithvi.model, x_prithvi)
forward_seconds = time.perf_counter() - started

{
    'forward_seconds': forward_seconds,
    'feature_shape': tuple(feature_map.shape),
    'feature_dtype': str(feature_map.dtype),
    'feature_device': str(feature_map.device),
}

## Optional One-Batch Head Training

Run this cell when the forward pass succeeds. It calls the script path, writes a JSONL result, and uses only one train batch plus one validation batch by default.

In [ ]:
cmd = [
    sys.executable,
    'scripts/run_prithvi_mac_pilot.py',
    '--device', device,
    '--label-budget', '0.10',
    '--batch-size', '1',
    '--image-size', '256',
    '--prithvi-image-size', str(PRITHVI_IMAGE_SIZE),
    '--limit-train-batches', '1',
    '--limit-val-batches', '1',
    '--output', 'artifacts/prithvi_mac_pilot/results.jsonl',
]

print(' '.join(cmd))
# result = subprocess.run(cmd, check=True, text=True, capture_output=True)
# print(result.stdout)

Uncomment the last two lines in the previous cell when you are ready to run the one-batch training smoke test. Keeping it commented makes the notebook safer to step through while you are verifying model loading and memory first.

## TerraMind Status Check

In [ ]:
try:
    import terratorch
    terramind_status = {
        'terratorch_installed': True,
        'terratorch_version': getattr(terratorch, '__version__', 'unknown'),
        'next_step': 'Use BACKBONE_REGISTRY.build for TerraMind base and large.',
    }
except Exception as exc:
    terramind_status = {
        'terratorch_installed': False,
        'error': f'{type(exc).__name__}: {exc}',
        'next_step': 'Install and validate TerraTorch before TerraMind end-to-end runs.',
    }

terramind_status